# 🧠 Notebook 1: Machine Learning Foundations for Mental Health Research

**Workshop:** Practical Introduction to AI for Mental Health Applications  
**Estimated time:** 25 minutes  
**Prerequisites:** None — just run each cell top to bottom!

## Learning Objectives
By the end of this notebook, you will be able to:
1. Load and explore a realistic clinical dataset
2. Preprocess data for machine learning (handle missing values, scale features)
3. Train two ML models (Logistic Regression and Random Forest) 
4. Evaluate model performance using clinically relevant metrics
5. Interpret model results using visualizations

---
> 💡 **How to use this notebook:** Click on each cell and press `Shift + Enter` to run it. Green cells with `# EXERCISE` are for you to modify. Blue cells with `# INSTRUCTOR NOTE` are pause points for group discussion.

## 0. Setup & Imports
First, we install and import all the Python libraries we need. This takes about 60 seconds — run it and grab a coffee ☕

In [ ]:
# Install required packages (only needed in Google Colab)
!pip install shap --quiet
!pip install seaborn --quiet

# Core data science libraries
import numpy as np          # Numerical computing
import pandas as pd         # Data manipulation and analysis
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns       # Statistical visualizations
import warnings
warnings.filterwarnings('ignore')

# Machine learning (scikit-learn)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc, roc_auc_score)
import joblib  # For saving our trained model

# WHY: Set random seeds everywhere for reproducibility
# This ensures everyone in the workshop gets identical results
np.random.seed(42)

# Make plots look nice
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✓ Setup complete! All libraries loaded successfully.")
print(f"  NumPy version: {np.__version__}")
print(f"  Pandas version: {pd.__version__}")

## 1. Loading & Exploring Our Dataset

### What is ABIDE?
The **Autism Brain Imaging Data Exchange (ABIDE)** is a real, freely available dataset that aggregates neuroimaging and phenotypic data from autism research sites worldwide. In real life, you can download it from [fcon_1000.projects.nitrc.org/indi/abide](http://fcon_1000.projects.nitrc.org/indi/abide).

**Why ASD prediction?**  
ASD is typically diagnosed between ages 2–5, but many children — especially girls and those with average/high IQ — remain undiagnosed until adolescence or adulthood. ML tools that flag at-risk youth from behavioral assessment scores could support earlier referral for formal diagnosis.

**Today's dataset:** Since live data downloads require authentication, we'll generate a **realistic synthetic dataset** that mirrors the statistical properties of actual ABIDE phenotypic data. All values are generated from distributions matching the real data — nothing here represents a real person.

### Features we'll use:
| Feature | Description | Range |
|---------|-------------|-------|
| AGE_AT_SCAN | Age in years | 6–58 |
| SEX | Binary sex | 0=Male, 1=Female |
| FIQ | Full-scale IQ | 70–145 |
| VIQ | Verbal IQ | 70–145 |
| PIQ | Performance IQ | 70–145 |
| SRS_RAW_TOTAL | Social Responsiveness Scale (higher = more symptoms) | 0–195 |
| ADOS_TOTAL | Autism Diagnostic Observation Schedule | 0–28 |
| SCQ_TOTAL | Social Communication Questionnaire | 0–39 |
| VINELAND_COMMUNICATION_STANDARD_SCORE | Adaptive behavior (higher = better) | 40–160 |
| **DX_GROUP** | **Our target: 1=Autism, 0=Control** | 0/1 |

In [ ]:
%%time
# Generate synthetic ABIDE-like dataset
# WHY: Synthetic data lets us work without internet and ensures no real patient data is used

np.random.seed(42)
N_ASD = 250    # Number of autism participants
N_CTRL = 250   # Number of control participants
N = N_ASD + N_CTRL

def generate_correlated_iq(n, mean_fiq, std_fiq=12):
    """Generate correlated FIQ, VIQ, PIQ scores using Cholesky decomposition.
    In real data, these three IQ measures correlate at r≈0.7 with each other."""
    # Correlation matrix for [FIQ, VIQ, PIQ]
    corr = np.array([[1.0, 0.7, 0.7],
                     [0.7, 1.0, 0.5],
                     [0.7, 0.5, 1.0]])
    # Cholesky decomposition creates correlated samples from uncorrelated normals
    L = np.linalg.cholesky(corr)
    uncorrelated = np.random.normal(0, 1, (n, 3))
    correlated = uncorrelated @ L.T
    # Scale to desired mean and std, then clip to realistic range
    scores = correlated * std_fiq + mean_fiq
    return np.clip(scores, 70, 145).round().astype(float)

# --- ASD Group ---
# WHY Beta(2,5): ABIDE participants skew toward adolescents/young adults rather than uniform across the lifespan
asd_age = 6 + 52 * np.random.beta(2, 5, N_ASD)
asd_sex = np.random.choice([0, 1], N_ASD, p=[0.82, 0.18])  # ASD is ~4:1 male:female
# WHY std_fiq=17: ASD shows greater IQ variability than controls; mean=102 reflects ABIDE I phenotypics
asd_iq = generate_correlated_iq(N_ASD, mean_fiq=102, std_fiq=17)

# WHY mixture model: ASD is highly heterogeneous — two clinical sub-populations reflect real phenotypic variation
# High-functioning subgroup (~55%): moderate symptom scores, IQ typically preserved
# Higher-severity subgroup (~45%): more pronounced symptoms across all measures
asd_subgroup = np.random.choice([0, 1], N_ASD, p=[0.55, 0.45])
asd_srs = np.where(asd_subgroup == 0,
                   np.random.normal(88, 30, N_ASD),    # High-functioning: moderate SRS
                   np.random.normal(128, 28, N_ASD))   # Higher-severity: elevated SRS
asd_ados = np.where(asd_subgroup == 0,
                    np.random.normal(8, 4, N_ASD),     # High-functioning: lower ADOS
                    np.random.normal(14, 4, N_ASD))    # Higher-severity: higher ADOS
asd_scq = np.random.normal(16, 6, N_ASD)
asd_vineland = np.random.normal(88, 19, N_ASD)

# --- Age compensation effect ---
# WHY: Older individuals with ASD often develop compensatory strategies that reduce apparent symptom
# severity on behavioural rating scales — a well-documented clinical phenomenon ("camouflaging")
age_compensation = (asd_age - 6) / 52 * 15  # up to ~15-point reduction at oldest ages
asd_srs -= age_compensation * np.random.uniform(0.5, 1.5, N_ASD)

# --- IQ masking effect ---
# WHY: Higher IQ in ASD correlates with better compensation on social measures; creates
# a non-linear interaction that linear models struggle to capture
iq_masking = (asd_iq[:, 0] - 85) / 60 * 12  # up to 12-point reduction at IQ=145
asd_srs -= np.maximum(iq_masking, 0)
asd_ados -= np.maximum(iq_masking / 6, 0)

# --- Correlated measurement noise ---
# WHY: Rater effects and testing-day variance create correlated noise across behavioural measures;
# this is a realistic source of within-session variability in clinical assessments
shared_noise_asd = np.random.normal(0, 8, N_ASD)
asd_srs += shared_noise_asd * 1.5
asd_scq += shared_noise_asd * 0.3
asd_vineland -= shared_noise_asd * 0.8

# Clip all ASD measures to valid clinical ranges
asd_srs = np.clip(asd_srs, 0, 195)
asd_ados = np.clip(asd_ados, 0, 28).round()
asd_scq = np.clip(asd_scq, 0, 39).round()
asd_vineland = np.clip(asd_vineland, 40, 160).round()

# --- Control Group ---
ctrl_age = 6 + 52 * np.random.beta(2, 5, N_CTRL)
ctrl_sex = np.random.choice([0, 1], N_CTRL, p=[0.55, 0.45])  # More balanced in controls
ctrl_iq = generate_correlated_iq(N_CTRL, mean_fiq=112, std_fiq=14)
# WHY wider SDs: controls in research settings show more phenotypic variability; some carry
# subthreshold autistic traits or have developmental concerns that prompted referral
ctrl_srs = np.clip(np.random.normal(48, 26, N_CTRL), 0, 195)
# WHY SD=3.5 for ADOS: some controls present near the clinical threshold due to social anxiety,
# ADHD, or other conditions that overlap with ASD symptom profiles
ctrl_ados = np.clip(np.random.normal(3.5, 3.5, N_CTRL), 0, 15).round()
ctrl_scq = np.clip(np.random.normal(7, 5, N_CTRL), 0, 39).round()
ctrl_vineland = np.clip(np.random.normal(108, 16, N_CTRL), 40, 160).round()

# Combine into a DataFrame
df = pd.DataFrame({
    'AGE_AT_SCAN': np.concatenate([asd_age, ctrl_age]),
    'SEX': np.concatenate([asd_sex, ctrl_sex]),
    'FIQ': np.concatenate([asd_iq[:, 0], ctrl_iq[:, 0]]),
    'VIQ': np.concatenate([asd_iq[:, 1], ctrl_iq[:, 1]]),
    'PIQ': np.concatenate([asd_iq[:, 2], ctrl_iq[:, 2]]),
    'SRS_RAW_TOTAL': np.concatenate([asd_srs, ctrl_srs]),
    'ADOS_TOTAL': np.concatenate([asd_ados, ctrl_ados]),
    'SCQ_TOTAL': np.concatenate([asd_scq, ctrl_scq]),
    'VINELAND_COMMUNICATION_STANDARD_SCORE': np.concatenate([asd_vineland, ctrl_vineland]),
    'DX_GROUP': np.concatenate([np.ones(N_ASD), np.zeros(N_CTRL)])  # 1=ASD, 0=Control
})

# Shuffle the dataset so ASD and Control aren't all together
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset created: {len(df)} participants")
print(f"  ASD: {int(df.DX_GROUP.sum())} participants")
print(f"  Control: {int((df.DX_GROUP == 0).sum())} participants")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Quick data overview
print("=== Dataset Info ===")
print(df.info())
print("\n=== Summary Statistics ===")
df.describe().round(2)

## 2. Exploratory Data Analysis (EDA)

Before training any model, we **always** explore our data visually. This helps us:
- Understand the distribution of each feature
- Spot class imbalance (more controls than ASD cases?)
- Identify which features might be most predictive
- Catch data quality issues early

> 💬 **DISCUSSION POINT:** What patterns do you expect to see between the ASD and Control groups? Which features do you think will be most different?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: Age distribution by diagnosis group ---
ax = axes[0, 0]
ax.hist(df[df.DX_GROUP == 1]['AGE_AT_SCAN'], alpha=0.6, bins=20,
        color='#B93680', label='ASD', density=True)
ax.hist(df[df.DX_GROUP == 0]['AGE_AT_SCAN'], alpha=0.6, bins=20,
        color='#2563EB', label='Control', density=True)
ax.set_xlabel('Age at Scan (years)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Age Distribution by Diagnosis Group', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# --- Plot 2: SRS score violin plot by group ---
ax = axes[0, 1]
plot_data = df[['SRS_RAW_TOTAL', 'DX_GROUP']].copy()
plot_data['Group'] = plot_data['DX_GROUP'].map({1: 'ASD', 0: 'Control'})
parts = ax.violinplot([df[df.DX_GROUP == 1]['SRS_RAW_TOTAL'],
                        df[df.DX_GROUP == 0]['SRS_RAW_TOTAL']],
                       positions=[1, 2], showmeans=True, showmedians=True)
parts['bodies'][0].set_facecolor('#B93680')
parts['bodies'][1].set_facecolor('#2563EB')
ax.set_xticks([1, 2])
ax.set_xticklabels(['ASD', 'Control'])
ax.set_ylabel('SRS Raw Total Score', fontsize=12)
ax.set_title('SRS Score Distribution by Group\n(Higher = More Autism Symptoms)',
             fontsize=14, fontweight='bold')

# --- Plot 3: Correlation heatmap ---
ax = axes[1, 0]
feature_cols = ['AGE_AT_SCAN', 'FIQ', 'SRS_RAW_TOTAL', 'ADOS_TOTAL',
                'SCQ_TOTAL', 'VINELAND_COMMUNICATION_STANDARD_SCORE', 'DX_GROUP']
corr_matrix = df[feature_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix\n(DX_GROUP = our target)',
             fontsize=14, fontweight='bold')

# --- Plot 4: Class balance ---
ax = axes[1, 1]
counts = df['DX_GROUP'].value_counts()
wedges, texts, autotexts = ax.pie(counts, labels=['ASD', 'Control'],
                                   colors=['#B93680', '#2563EB'],
                                   autopct='%1.1f%%', startangle=90,
                                   textprops={'fontsize': 12})
ax.set_title('Class Balance\n(Good: roughly 50/50)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ EDA plots saved as 'eda_overview.png'")

> 🔧 **EXERCISE:** The violin plot above shows SRS scores. Try changing it to show ADOS scores instead.
> 
> **Hint:** Replace `'SRS_RAW_TOTAL'` with `'ADOS_TOTAL'` in the violin plot code above. Also update the y-axis label and title. What do you notice about the ADOS score separation between groups?

## 3. Preprocessing

Real clinical datasets are messy. Before training any model, we need to:
1. **Handle missing values** — ML algorithms can't work with NaN values
2. **Scale features** — Some algorithms assume features are on similar scales
3. **Split data** — Never test on data the model has seen!

> 🔑 **Key principle:** All preprocessing decisions must be made using ONLY training data, then applied to test data. Otherwise, we "leak" information from the test set into training.

In [ ]:
# WHY: We introduce realistic missing data (~5%) to simulate what real datasets look like
# In real ABIDE data, not all sites collected all assessments
np.random.seed(42)
df_messy = df.copy()
missing_cols = ['SRS_RAW_TOTAL', 'ADOS_TOTAL', 'SCQ_TOTAL', 'VINELAND_COMMUNICATION_STANDARD_SCORE']
for col in missing_cols:
    # Randomly set 5% of values to NaN (Not a Number = missing)
    missing_mask = np.random.random(len(df_messy)) < 0.05
    df_messy.loc[missing_mask, col] = np.nan

print(f"Missing values introduced:")
print(df_messy.isnull().sum())
print(f"\nTotal missing values: {df_messy.isnull().sum().sum()}")
print(f"That's {df_messy.isnull().sum().sum() / df_messy.size * 100:.1f}% of the dataset")

In [ ]:
# Define features (X) and target (y)
feature_cols = ['AGE_AT_SCAN', 'SEX', 'FIQ', 'VIQ', 'PIQ',
                'SRS_RAW_TOTAL', 'ADOS_TOTAL', 'SCQ_TOTAL',
                'VINELAND_COMMUNICATION_STANDARD_SCORE']

X = df_messy[feature_cols]
y = df_messy['DX_GROUP'].astype(int)

# --- Step 1: Train/Test Split ---
# WHY stratify=y: ensures both splits have same proportion of ASD/Control
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} participants")
print(f"Test set:     {X_test.shape[0]} participants")
print(f"Train ASD%:   {y_train.mean():.1%}")
print(f"Test ASD%:    {y_test.mean():.1%}")

# --- Step 2: Handle Missing Values ---
# WHY: Fit imputer on TRAIN only, then transform both train and test
# (avoids "data leakage" where test set information influences training)
imputer = SimpleImputer(strategy='median')  # Use median (robust to outliers)
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)  # Note: only .transform(), not .fit_transform()

# --- Step 3: Scale Features ---
# WHY: Logistic regression assumes features are on similar scales
# StandardScaler: transforms each feature to mean=0, std=1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Convert back to DataFrame for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)

print("\n✓ Preprocessing complete!")
print(f"Feature means after scaling (should be ≈0):")
print(X_train_scaled.mean().round(3))

> 🔧 **EXERCISE:** Try changing `test_size=0.2` to `test_size=0.3` in the train/test split above.
> 
> **Hint:** Re-run the cell. How does the number of training and test participants change? Does the ASD percentage stay the same? (It should, because of `stratify=y`.)

## 4. Training Our First Model — Logistic Regression

**Logistic Regression** is the "Hello World" of classification models. Despite its name, it predicts a *probability* that a sample belongs to a class (here: probability of ASD).

**The intuition:** It learns a weighted combination of features. A high SRS score *increases* the probability of ASD; a high Vineland score *decreases* it. These weights are what the model "learns" from data.

> 💬 **INSTRUCTOR NOTE:** Before running this cell, ask the group: "If you had to predict ASD from just one of these features, which would you pick? Why?"

In [ ]:
%%time
# Train Logistic Regression
# WHY max_iter=1000: default 100 iterations sometimes isn't enough for convergence
lr_model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_model.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]  # Probability of ASD

# Print performance report
print("=== Logistic Regression Performance ===")
print(classification_report(y_test, y_pred_lr, target_names=['Control', 'ASD']))

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Pred: Control', 'Pred: ASD'],
            yticklabels=['True: Control', 'True: ASD'])
ax.set_title('Logistic Regression — Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix_lr.png', dpi=150, bbox_inches='tight')
plt.show()

> 💬 **DISCUSSION POINT:** Look at the confusion matrix. 
> - **True Positives (bottom-right):** ASD cases correctly identified (our "hits")
> - **False Negatives (bottom-left):** ASD cases MISSED by the model (dangerous in screening!)
> - **False Positives (top-right):** Controls incorrectly flagged as ASD (costly — unnecessary referrals)
>
> In a **screening tool**, we want to minimize False Negatives (catch everyone who might have ASD). In a **diagnostic tool**, we want to minimize False Positives (only flag when confident). Which is more important here?

## 5. A More Powerful Model — Random Forest

**Random Forest** is an *ensemble* method — it combines many simple decision trees to make a robust prediction.

**The intuition:** 
- Each decision tree asks a series of yes/no questions ("Is SRS > 80? → If yes, is ADOS > 10? → ...")
- Random Forest trains 100 different trees, each on a random subset of data and features
- Final prediction = majority vote across all trees

**Why it works well for clinical data:** Handles non-linear relationships, works without feature scaling, naturally handles interactions between features, and provides feature importance rankings.

In [ ]:
%%time
# Train Random Forest
# WHY n_estimators=100: 100 trees gives good performance without being too slow
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

print("=== Random Forest Performance ===")
print(classification_report(y_test, y_pred_rf, target_names=['Control', 'ASD']))

# Feature importance plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ax = axes[0]
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Pred: Control', 'Pred: ASD'],
            yticklabels=['True: Control', 'True: ASD'])
ax.set_title('Random Forest — Confusion Matrix', fontsize=14, fontweight='bold')

# Feature importances (how much each feature contributes to predictions)
ax = axes[1]
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances_sorted = importances.sort_values(ascending=True)
colors = ['#B93680' if imp > importances.median() else '#6B7280'
          for imp in importances_sorted.values]
importances_sorted.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Feature Importances\n(Which features matter most?)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score', fontsize=12)
ax.axvline(x=importances.median(), color='black', linestyle='--', alpha=0.5, label='Median')
ax.legend()

plt.tight_layout()
plt.savefig('rf_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTop 3 most important features:")
for feat, imp in importances.sort_values(ascending=False).head(3).items():
    print(f"  {feat}: {imp:.3f}")

> 🔧 **EXERCISE:** Try changing `n_estimators=100` to `n_estimators=10` in the Random Forest above.
> 
> **Hint:** Re-run the cell. What happens to the performance metrics? What about the time taken (shown by `%%time`)? This illustrates the trade-off between model complexity and computational cost.

## 6. Understanding Model Performance — The ROC Curve

The **ROC (Receiver Operating Characteristic) curve** shows how a model's sensitivity (recall) and specificity trade off against each other at different classification thresholds.

- **AUC = 1.0:** Perfect model
- **AUC = 0.5:** Random guessing (no better than a coin flip)
- **AUC > 0.8:** Generally considered good for clinical screening tools

**Threshold selection:** By default, models predict "ASD" when probability > 0.5. But we can choose different thresholds depending on our clinical priorities.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

# ROC curves for both models
for name, y_prob in [('Logistic Regression', y_prob_lr), ('Random Forest', y_prob_rf)]:
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_score = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2.5, label=f'{name} (AUC = {auc_score:.3f})')

# Diagonal line = random classifier
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6, label='Random Classifier (AUC = 0.50)')

# Annotate a clinically useful operating point on RF curve
fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_prob_rf)
# Find threshold closest to sensitivity=0.85 (good for screening)
idx = np.argmin(np.abs(tpr_rf - 0.85))
ax.scatter(fpr_rf[idx], tpr_rf[idx], s=150, zorder=5, color='#B93680',
           label=f'Operating point: Sens={tpr_rf[idx]:.2f}, Spec={1-fpr_rf[idx]:.2f}')
ax.annotate(f'Threshold={thresholds_rf[idx]:.2f}\n(85% sensitivity)',
            xy=(fpr_rf[idx], tpr_rf[idx]), xytext=(fpr_rf[idx]+0.1, tpr_rf[idx]-0.1),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=13)
ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=13)
ax.set_title('ROC Curves: Logistic Regression vs Random Forest', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("Interpretation:")
print(f"  LR AUC: {roc_auc_score(y_test, y_prob_lr):.3f}")
print(f"  RF AUC: {roc_auc_score(y_test, y_prob_rf):.3f}")
print("\nFor a SCREENING tool, we'd choose the operating point with ~85% sensitivity")
print("(catch most ASD cases) even if it means some false positives.")

> 💬 **DISCUSSION POINT:** In **screening** (e.g., population-level ASD surveillance), we want HIGH sensitivity — we'd rather refer 10 children who don't have ASD than miss one who does. In **diagnosis**, we want HIGH specificity — we only want to label a child as ASD when we're confident.
>
> How would you set the threshold differently for a screening tool vs. a diagnostic aid?

## 7. Cross-Validation — Making Results Trustworthy

One train/test split can be "lucky" or "unlucky." **Cross-validation** gives us a more reliable performance estimate by training and evaluating on 5 different splits.

In [ ]:
%%time
# 5-fold stratified cross-validation
# WHY StratifiedKFold: ensures each fold has the same ASD/Control ratio
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# We evaluate on the full scaled dataset (re-imputed without leakage)
X_full_imputed = imputer.fit_transform(X)
X_full_scaled = scaler.fit_transform(X_full_imputed)
X_full_scaled = pd.DataFrame(X_full_scaled, columns=feature_cols)

results = {}
for name, model in [('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
                     ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))]:
    cv_results = cross_validate(model, X_full_scaled, y, cv=cv,
                                scoring=['accuracy', 'f1', 'roc_auc'],
                                return_train_score=False)
    results[name] = {
        'Accuracy': (cv_results['test_accuracy'].mean(), cv_results['test_accuracy'].std()),
        'F1': (cv_results['test_f1'].mean(), cv_results['test_f1'].std()),
        'AUC': (cv_results['test_roc_auc'].mean(), cv_results['test_roc_auc'].std())
    }
    print(f"\n{name}:")
    for metric, (mean, std) in results[name].items():
        print(f"  {metric}: {mean:.3f} \u00b1 {std:.3f}")

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['Accuracy', 'F1', 'AUC']
colors = {'Logistic Regression': '#2563EB', 'Random Forest': '#B93680'}

for i, metric in enumerate(metrics):
    ax = axes[i]
    models = list(results.keys())
    means = [results[m][metric][0] for m in models]
    stds = [results[m][metric][1] for m in models]
    bars = ax.bar(models, means, yerr=stds, capsize=8,
                  color=[colors[m] for m in models], alpha=0.85, error_kw={'linewidth': 2})
    ax.set_title(f'{metric}\n(5-fold CV, mean \u00b1 std)', fontsize=13, fontweight='bold')
    ax.set_ylim(0.5, 1.05)
    ax.set_ylabel(metric, fontsize=11)
    ax.tick_params(axis='x', rotation=15)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{mean:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('cross_validation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

> 🔧 **EXERCISE:** Try changing `n_splits=5` to `n_splits=10` in the StratifiedKFold above.
>
> **Hint:** Re-run the cell. Does the mean performance change much? Does the standard deviation (uncertainty) change? With k=10, each fold uses more training data but the test folds are smaller.

## 8. Summary & Bridge to Notebook 2

### What we learned:
- ✅ How to generate and explore a realistic clinical dataset
- ✅ How to preprocess data (imputation, scaling, train/test split)
- ✅ How to train and evaluate Logistic Regression and Random Forest classifiers
- ✅ How to interpret AUC-ROC, confusion matrices, and feature importances
- ✅ Why cross-validation gives more reliable performance estimates

### Key takeaway:
Random Forest outperforms Logistic Regression on this task — and the gap is meaningful. The dataset reflects real clinical complexity: ASD is a heterogeneous condition where symptom severity is modulated by age (compensation), IQ (masking), and within-person variability. These non-linear interactions are exactly what ensemble methods excel at capturing, while a single linear boundary cannot.

This is also why **AUC in the 0.80s is realistic** for a well-built ASD screening tool. Perfect classification would be a red flag — it would likely mean our features are proxies for diagnosis itself, not genuine predictors.

**But here's the critical next question:** *Why* does the model make each prediction? When it flags a child as likely having ASD, which features drove that decision? Could it be picking up on spurious correlations?

→ **This is what Notebook 2 answers using SHAP (Shapley Additive Explanations).**

In [ ]:
# Save the trained Random Forest model for use in Notebook 2
# WHY joblib: more efficient than pickle for scikit-learn models (handles numpy arrays better)
joblib.dump(rf_model, 'rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(imputer, 'imputer.pkl')

# Also save test data for Notebook 2
X_test_scaled.to_csv('X_test.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
y_prob_rf_series = pd.Series(y_prob_rf, name='prob_ASD')
y_prob_rf_series.to_csv('y_prob_rf.csv', index=False)

print("✓ Saved:")
print("  rf_model.pkl   — trained Random Forest model")
print("  scaler.pkl     — fitted StandardScaler")
print("  imputer.pkl    — fitted median imputer")
print("  X_test.csv     — test features (scaled)")
print("  y_test.csv     — test labels")
print("  y_prob_rf.csv  — model probability scores")
print("\n→ Open Notebook 2 to explain your model with SHAP!")